In [2]:
!pip install pyspark delta-spark

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("DeltaLakeFullDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.0 MB/s eta 0:00:00


In [3]:
data = [
    (1, "Amit", "Laptop", 50000),
    (2, "Riya", "Phone", 20000),
    (3, "John", "Tablet", 15000)
]

columns = ["id", "name", "product", "sales"]
df = spark.createDataFrame(data, columns)
df.show()

+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  1|Amit| Laptop|50000|
|  2|Riya|  Phone|20000|
|  3|John| Tablet|15000|
+---+----+-------+-----+



In [4]:
# Write to Delta
df.write.format("delta").mode("overwrite").save("/content/delta-table")

In [5]:
delta_df = spark.read.format("delta").load("/content/delta-table")
delta_df.show()

+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  2|Riya|  Phone|20000|
|  3|John| Tablet|15000|
|  1|Amit| Laptop|50000|
+---+----+-------+-----+



In [6]:
new_data = [
    (4, "Amit", "Phone", 25000),
    (5, "Riya", "Laptop", 60000)
]

new_df = spark.createDataFrame(new_data, columns)
new_df.write.format("delta").mode("append").save("/content/delta-table")

In [7]:
# Update using DeltaTable API
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/content/delta-table")

delta_table.update(
    condition="name = 'Amit'",
    set={"sales": "sales + 5000"}
)

delta_table.toDF().show()

+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  2|Riya|  Phone|20000|
|  3|John| Tablet|15000|
|  1|Amit| Laptop|55000|
|  5|Riya| Laptop|60000|
|  4|Amit|  Phone|30000|
+---+----+-------+-----+



In [8]:
# Delete records
delta_table.delete("sales < 20000")

In [9]:
delta_table.toDF().show()

+---+----+-------+-----+
| id|name|product|sales|
+---+----+-------+-----+
|  1|Amit| Laptop|55000|
|  5|Riya| Laptop|60000|
|  2|Riya|  Phone|20000|
|  4|Amit|  Phone|30000|
+---+----+-------+-----+



In [11]:
# Schema Enforcement Example (This will trigger an error)
wrong_data = [
    (6, "Sara", "Camera", "INVALID") # sales should be int, but "INVALID" is a string
]

wrong_df = spark.createDataFrame(wrong_data, columns)

In [12]:
# This line causes AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS]
wrong_df.write.format("delta").mode("append").save("/content/delta-table")

AnalysisException: [DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'sales' and 'sales'

In [ ]:
#Time Travel (Version history)
# Accessing version 0 of the table
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/content/delta-table") \
    .show()